# Platform graph build — corpus → NER/NEL → Layer A/B/C → Elasticsearch + Neo4j

Builds the FoodScholar hierarchical graph over the **whole platform corpus** and
persists it into the platform's own stores. One linear pass, resumable at every
phase, with the NER/NEL step parallelised across processes.

| # | Phase | What it writes | Parallel |
|---|-------|----------------|----------|
| 1 | Configure | — | — |
| 2 | Corpus: textbooks + guides | *(reused, not re-chunked)* | — |
| 3 | Corpus: article abstracts | corpus CSV | — |
| 4 | Load chunks | ES chunk index | — |
| 5 | Warm the NEL index | HNSW index on disk | — |
| 6 | **NER + NEL + embed** | ES chunk index | **yes** |
| 7 | Entities | ES entity index | — |
| 8 | Layer A — shelves | Neo4j | — |
| 9 | Attach | ES + Neo4j | — |
| 10 | Layer B — themes | Neo4j (+ Groq) | — |
| 11 | Layer C — cards | Neo4j + ES (+ Groq) | — |
| 12 | Verify & publish | — | — |

**Resumability.** Every phase is idempotent and skips completed work. Phase 6
selects only chunks whose `enrichment_version` is not yet `annotate-v2`, so an
interrupted run resumes by re-executing the same cell. Nothing here needs to be
run twice to be correct, and nothing is destructive except where it says so.

**What this notebook will not do.** It never re-chunks the textbook and guide
corpus. Chunk ids are fresh UUIDs under the default strategy, so re-chunking
assigns new ones and orphans every card, relation and attachment citing the old
ones — the library guards `chunk_documents` with `force=True` for exactly this
reason. Those 13,344 chunks already exist in the platform index; they are
reused as they are.

## 0. Preflight

Secrets come from the environment, never from this file. Run with:

```bash
export ELASTICSEARCH_URL=http://localhost:9200      # port-forward or in-cluster
export NEO4J_URL=bolt://localhost:7687
export NEO4J_PASSWORD=...                           # the user/password half of NEO4J_AUTH
export GROQ_API_KEY=...                             # Layer B labels + Layer C cards
export FS_WORKERS=4                                 # annotate processes
export FS_WORKER_THREADS=2                          # torch threads inside each
```

If the stores are inside the cluster, forward them first — both are `ClusterIP`
with no ingress:

```bash
kubectl --context k8s-w -n wf-prod port-forward svc/elastic 9200:9200 &
kubectl --context k8s-w -n wf-prod port-forward svc/neo4j   7687:7687 &
```

In [ ]:
import os, sys, time, json, math
from pathlib import Path

REQUIRED = ["ELASTICSEARCH_URL", "NEO4J_URL", "NEO4J_PASSWORD", "GROQ_API_KEY"]
missing = [v for v in REQUIRED if not os.environ.get(v)]
if missing:
    raise SystemExit(f"missing environment: {', '.join(missing)}")

ROOT       = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA       = ROOT / "data"
CORPUS_DIR = DATA / "corpus_platform"
CORPUS_DIR.mkdir(parents=True, exist_ok=True)

ES_URL   = os.environ["ELASTICSEARCH_URL"]
NEO4J_URL = os.environ["NEO4J_URL"]

# Index names on the platform. The chunk and card indices are the ones the
# FoodScholar API already reads; `articles` is the source corpus.
CHUNK_INDEX   = os.environ.get("FS_CHUNK_INDEX", "foodscholar_chunks")
CARD_INDEX    = os.environ.get("FS_CARD_INDEX", "foodscholar_cards")
ARTICLE_INDEX = os.environ.get("FS_ARTICLE_INDEX", "articles")

WORKERS = int(os.environ.get("FS_WORKERS", "4"))

print(f"root    {ROOT}")
print(f"elastic {ES_URL}")
print(f"neo4j   {NEO4J_URL}")
print(f"workers {WORKERS}")

## 1. Configuration

One config dict, used by this process **and** by every annotate worker — they
build their own `FoodScholar` from it, so there is exactly one description of
where the data lives.

Two notes on the stores:

* `annotate.batch_size` is 16 by default, which is tuned for CPU. On a GPU
  raise it until the card is busy; it is the single biggest lever on phase 6.
* The library's Neo4j store opens `session()` with **no database argument**, so
  everything lands in the default `neo4j` database — the same one holding
  RecipeWrangler's graph. The labels are disjoint (`:Shelf` / `:Theme` /
  `:Card` versus `:Recipe` / `:Ingredient`), so they coexist, but this build is
  **not** isolated from it. Point `NEO4J_URL` at a separate instance if you
  want that isolation today.

In [ ]:
from foodscholar import FoodScholar

CONFIG = {
    "corpus": {
        "chunks_path": str(CORPUS_DIR / "chunks.parquet"),
        # Deliberately unset: the snapshot short-circuit would skip phase 6
        # entirely on a resume, which is the opposite of what we want here.
        # Resumption is handled by the enrichment_version filter instead.
        "annotated_snapshot_path": None,
    },
    "ontology": {
        "foodon_path": str(DATA / "foodon.owl"),
        "cache_path": str(DATA / "foodon_cache.parquet"),
        "include_imports": False,
    },
    "annotate": {
        "ner": "gliner",
        "batch_size": int(os.environ.get("FS_ANNOTATE_BATCH", "16")),
        "linker": {
            "nel_backend": "hnsw",
            "nel_encoder": "biolord",
            # Pinned rather than content-addressed, so every worker loads the
            # same file instead of each deriving a path and rebuilding it.
            "nel_index_path": str(DATA / "foodon_hnsw_biolord.bin"),
            "nel_metadata_path": str(DATA / "foodon_hnsw_biolord.meta.json"),
        },
    },
    "storage": {
        "chunk_store": {"backend": "elastic", "url": ES_URL, "index": CHUNK_INDEX},
        "card_store":  {"backend": "elastic", "url": ES_URL, "index": CARD_INDEX},
        "graph_store": {
            "backend": "neo4j",
            "url": NEO4J_URL,
            "user": os.environ.get("NEO4J_USER", "neo4j"),
            "password": os.environ["NEO4J_PASSWORD"],
        },
    },
    "llm": {"provider": "groq", "model": os.environ.get("FS_LLM_MODEL", "openai/gpt-oss-120b")},
}

fs = FoodScholar.from_config(CONFIG)
print(json.dumps(fs.info(), indent=2, default=str))
print("config hash:", fs.config_hash)

## 2. Corpus — textbooks and guides (reused, never re-chunked)

These 13,344 chunks are already in the platform index with stable ids. This
cell only *verifies* them. If it reports zero, the chunk index is empty and you
should load `data/annotated.parquet` rather than re-chunk the PDFs.

In [ ]:
from elasticsearch import Elasticsearch

es = Elasticsearch(hosts=ES_URL)

def count(index, body=None):
    return int(es.count(index=index, body=body)["count"]) if es.indices.exists(index=index) else 0

by_source = {}
if es.indices.exists(index=CHUNK_INDEX):
    aggs = es.search(index=CHUNK_INDEX, size=0,
                     body={"aggs": {"s": {"terms": {"field": "source_type", "size": 10}}}})
    by_source = {b["key"]: b["doc_count"] for b in aggs["aggregations"]["s"]["buckets"]}

print("chunks already in the platform index, by source_type:")
for k, v in sorted(by_source.items()):
    print(f"  {k:10s} {v:7d}")
print(f"  {'TOTAL':10s} {count(CHUNK_INDEX):7d}")

if not by_source.get("textbook") and not by_source.get("guide"):
    print("\n!! No textbook/guide chunks found. Load data/annotated.parquet into the")
    print("   chunk store before continuing — do NOT re-chunk the PDFs, it reassigns ids.")

## 3. Corpus — article abstracts from the platform index

The one piece the library has no CLI for: `chunk-corpus` takes PDFs only, and
abstracts go through `fs.chunk_texts()`. This pulls every article with a usable
abstract out of the `articles` index and hands it to the same 512/64 window the
PDFs use — one algorithm, two fine-unit producers (NLTK sentences here, Docling
for PDFs).

`source_doc_id` is the article URN, so a chunk can always be traced back to the
catalogue record it came from.

In [ ]:
MIN_ABSTRACT_CHARS = int(os.environ.get("FS_MIN_ABSTRACT_CHARS", "200"))
ARTICLE_LIMIT = int(os.environ.get("FS_ARTICLE_LIMIT", "0"))  # 0 = all; set small to rehearse

texts, meta = {}, {}
after, scanned, skipped_short = None, 0, 0

while True:
    body = {
        "size": 1000,
        "_source": ["urn", "id", "title", "abstract", "publication_year", "doi", "venue"],
        "query": {"exists": {"field": "abstract"}},
        "sort": [{"_id": "asc"}],
    }
    if after:
        body["search_after"] = after
    hits = es.search(index=ARTICLE_INDEX, body=body)["hits"]["hits"]
    if not hits:
        break
    for h in hits:
        s = h["_source"]
        doc_id = s.get("urn") or s.get("id") or h["_id"]
        abstract = (s.get("abstract") or "").strip()
        scanned += 1
        # A 40-character "abstract" is a placeholder, not evidence. Chunking it
        # produces a node with no content behind it and inflates support counts.
        if len(abstract) < MIN_ABSTRACT_CHARS:
            skipped_short += 1
            continue
        texts[doc_id] = abstract
        meta[doc_id] = {
            "title": s.get("title") or "",
            "year": s.get("publication_year") or "",
            "doi": s.get("doi") or "",
        }
    after = hits[-1]["sort"]
    if ARTICLE_LIMIT and len(texts) >= ARTICLE_LIMIT:
        break

print(f"scanned {scanned} articles; kept {len(texts)}; skipped {skipped_short} short/empty")

In [ ]:
abstracts_csv = CORPUS_DIR / "abstracts.csv"

if abstracts_csv.exists() and abstracts_csv.stat().st_size > 0:
    print(f"reusing {abstracts_csv} ({abstracts_csv.stat().st_size/1e6:.1f} MB) — delete it to re-chunk")
else:
    t0 = time.perf_counter()
    abstracts_csv = fs.chunk_texts(
        texts,
        out_path=abstracts_csv,
        source_type="abstract",
        metadata=meta,
    )
    print(f"wrote {abstracts_csv} in {time.perf_counter()-t0:.1f}s")

## 4. Load the abstract chunks into the store

Chunks only — **no annotation**. `load_chunks` upserts them with
`enrichment_version = "v0"`, which is precisely what phase 6 selects on.
Splitting load from annotate is what makes the expensive half restartable.

In [ ]:
before = count(CHUNK_INDEX)
fs.load_chunks(abstracts_csv)
es.indices.refresh(index=CHUNK_INDEX)
after_n = count(CHUNK_INDEX)
print(f"chunk index: {before} -> {after_n}  (+{after_n - before})")

## 5. Warm the NEL index — once, before any worker starts

`HNSWNELIndex` builds from the loaded ontology on first use and saves to
`nel_index_path`; on every later construction it loads that file. Building it
here, in one process, is what lets N workers start without N encodings of
FoodOn. Skip this and the first run of phase 6 has every worker build the same
index simultaneously.

In [ ]:
index_path = Path(CONFIG["annotate"]["linker"]["nel_index_path"])
t0 = time.perf_counter()
_ = fs.linker            # constructing it builds-and-saves, or loads
print(f"NEL index ready in {time.perf_counter()-t0:.1f}s")
print(f"  {index_path}  ({index_path.stat().st_size/1e6:.1f} MB)" if index_path.exists()
      else "  !! index file missing — workers will each rebuild it")

## 6. NER + NEL + embed, in parallel

The phase this notebook exists for. The library's annotate runner is already
batched — one `extract_batch`, one `link_many`, one `embed` per batch — but it
is a single sequential pass over the whole store. Here the corpus is sharded
and each shard runs the *same* runner in its own process.

**Sizing.** Each worker holds GLiNER (~1.5 GB), bge-base (~0.4 GB), the HNSW
index and the ontology: budget **3–4 GB of RAM per worker**. On a GPU, one or
two workers saturate it and more only add contention — raise
`FS_ANNOTATE_BATCH` instead. On CPU, workers are the lever and
`FS_WORKER_THREADS` keeps them from fighting over cores.

In [ ]:
%%writefile ../scripts/corpus/annotate_shard.py
"""One shard of the annotate phase, in its own process.

A module rather than a notebook closure on purpose: ``ProcessPoolExecutor``
pickles the callable by qualified name, and under the ``spawn`` start method a
function defined in a notebook cell cannot be re-imported by the child. Spawn
is not optional here — CUDA and ``fork`` do not mix, and a forked child that
touches an already-initialised CUDA context deadlocks or corrupts it.

Each worker is self-contained: it builds its own FoodScholar from the same
config dict, so it has its own GLiNER, its own linker and its own embedder.
That is the cost of the parallelism (roughly 3-4 GB of RAM per worker) and the
reason the NEL index is built once beforehand — workers load it from disk
instead of re-encoding FoodOn each.
"""
from __future__ import annotations

import os
from typing import Any


def annotate_shard(payload: tuple[dict[str, Any], list[str]]) -> dict[str, Any]:
    """Annotate one shard of chunk ids and write the results back to the store.

    Returns a summary rather than the chunks themselves: shipping annotated
    chunks back through the pool would pickle every 768-float embedding across
    a process boundary for no reason — the worker has already persisted them.
    """
    config_dict, chunk_ids = payload

    # Threads per worker. Left to its own devices torch grabs every core in
    # every process at once, and N workers each spawning N threads is slower
    # than one. Set before torch is imported, which is why this is at the top.
    os.environ.setdefault("OMP_NUM_THREADS", os.environ.get("FS_WORKER_THREADS", "2"))
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

    from foodscholar import FoodScholar
    from foodscholar.annotate import runner
    from foodscholar.storage.memory import InMemoryChunkStore

    fs = FoodScholar.from_config(config_dict)
    store = fs.chunk_store

    chunks = store.get_many(list(chunk_ids))
    if not chunks:
        return {"requested": len(chunk_ids), "annotated": 0, "mentions": 0, "links": 0}

    # The runner annotates everything in the store it is handed, so the shard
    # goes into a scratch store. This reuses the library's exact annotate path
    # — same batching, same NER, same linker — without reaching into it.
    scratch = InMemoryChunkStore()
    scratch.upsert(chunks)
    meta = runner.run(
        scratch,
        ner=fs.ner,
        linker=fs.linker,
        embedder=fs.embedder,
        config=fs.config,
    )

    annotated = scratch.scan()
    store.upsert(annotated)

    # Counted here rather than read off `meta`: ArtifactMeta carries
    # `record_count` and nothing else, so mention and link totals have to come
    # from the chunks. They are the phase's only quality signal — a shard that
    # annotated every chunk and linked none is a broken NEL index, not a
    # successful run.
    return {
        "requested": len(chunk_ids),
        "annotated": len(annotated),
        "mentions": sum(len(c.mentions or []) for c in annotated),
        "links": sum(len(c.entity_links or []) for c in annotated),
        "record_count": meta.record_count,
    }

In [ ]:
# Only what has not been annotated yet. Re-running this cell after an
# interruption picks up exactly where it stopped.
PENDING_QUERY = {"bool": {"must_not": [{"term": {"enrichment_version": "annotate-v2"}}]}}

es.indices.refresh(index=CHUNK_INDEX)
pending_total = count(CHUNK_INDEX, {"query": PENDING_QUERY})
print(f"pending: {pending_total} of {count(CHUNK_INDEX)} chunks")

pending_ids, after = [], None
while True:
    body = {"size": 5000, "_source": False, "query": PENDING_QUERY, "sort": [{"_id": "asc"}]}
    if after:
        body["search_after"] = after
    hits = es.search(index=CHUNK_INDEX, body=body)["hits"]["hits"]
    if not hits:
        break
    pending_ids.extend(h["_id"] for h in hits)
    after = hits[-1]["sort"]

print(f"collected {len(pending_ids)} ids to annotate")

In [ ]:
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed

sys.path.insert(0, str(ROOT / "scripts" / "corpus"))
from annotate_shard import annotate_shard

# spawn, not fork: CUDA and fork do not mix, and a forked child touching an
# initialised CUDA context deadlocks. Harmless on a CPU-only box.
ctx = mp.get_context("spawn")

SHARD_SIZE = int(os.environ.get("FS_SHARD_SIZE", "500"))
shards = [pending_ids[i:i + SHARD_SIZE] for i in range(0, len(pending_ids), SHARD_SIZE)]
print(f"{len(shards)} shards of <= {SHARD_SIZE} across {WORKERS} workers")

done = {"chunks": 0, "mentions": 0, "links": 0}
t0 = time.perf_counter()

if shards:
    with ProcessPoolExecutor(max_workers=WORKERS, mp_context=ctx) as pool:
        futures = {pool.submit(annotate_shard, (CONFIG, s)): i for i, s in enumerate(shards)}
        for n, fut in enumerate(as_completed(futures), 1):
            try:
                r = fut.result()
            except Exception as exc:
                # One bad shard must not lose the other N-1. The ids stay
                # pending, so re-running the cell retries just those.
                print(f"  shard {futures[fut]} FAILED: {exc}")
                continue
            done["chunks"] += r["annotated"]
            done["mentions"] += r["mentions"]
            done["links"] += r["links"]
            elapsed = time.perf_counter() - t0
            rate = done["chunks"] / elapsed if elapsed else 0
            remaining = (len(pending_ids) - done["chunks"]) / rate if rate else float("nan")
            print(f"  [{n}/{len(shards)}] {done['chunks']}/{len(pending_ids)} chunks "
                  f"| {rate:.1f}/s | eta {remaining/60:.1f} min", flush=True)

link_rate = done["links"] / done["mentions"] if done["mentions"] else 0
print(f"\nannotated {done['chunks']} chunks in {(time.perf_counter()-t0)/60:.1f} min")
print(f"mentions {done['mentions']}, links {done['links']}, link rate {link_rate:.1%}")

In [ ]:
# The check that matters: a run that annotated everything and linked nothing is
# a broken NEL index, not a successful pass.
es.indices.refresh(index=CHUNK_INDEX)
still_pending = count(CHUNK_INDEX, {"query": PENDING_QUERY})
linked = count(CHUNK_INDEX, {"query": {"exists": {"field": "foodon_ids"}}})
print(f"still pending : {still_pending}   (re-run the cell above if non-zero)")
print(f"chunks linked : {linked}")
assert still_pending == 0, "annotation incomplete — re-run the parallel cell"

## 6b. Embedding backstop


In [ ]:
# Insurance, not a phase. The annotate runner embeds every chunk it touches,
# so this should report nothing to do — but `only_missing=True` makes it free
# to assert that, and Layer B silently produces garbage from unembedded chunks.
meta = fs.embed(only_missing=True)
print(meta)


## 7. Entities

Collapses per-chunk links into the `Entity` records Layer A projects and
Layer B clusters. Single process — it is a global aggregation.

In [ ]:
meta = fs.build_entities()
print(meta)
print("entities:", len(fs.entities))

## 8. Layer A — the shelf backbone

In [ ]:
meta = fs.build_layer_a()
shelves = fs.graph_store.list_shelves()
by_facet = {}
for s in shelves:
    by_facet[s.facet] = by_facet.get(s.facet, 0) + 1
print(meta)
print(f"shelves: {len(shelves)}")
for f, n in sorted(by_facet.items()):
    print(f"  {f:20s} {n}")

## 9. Attach chunks to shelves

In [ ]:
meta = fs.attach()
print(meta)
print("chunks with a shelf:", count(CHUNK_INDEX, {"query": {"exists": {"field": "shelf_ids"}}}))

## 10. Layer B — themes

Clustering is deterministic (`random_state`), but theme **labels** come from one
LLM call each, so theme ids are not reproducible across runs. That is why any
rebuild orphans existing cards — expect to run Layer C after this, always.

**Stop here and look before continuing.** Layer B is where a bad corpus
decision shows up, and it is the last point at which stopping is free: Layer C
spends an LLM call per theme.

In [ ]:
FACETS = ["foods", "health", "sustainability", "dietary_patterns", "allergies", "nutrients"]

for facet in FACETS:
    t0 = time.perf_counter()
    try:
        meta = fs.build_layer_b(facet=facet)
        print(f"{facet:20s} {meta.record_count:5d} themes  ({time.perf_counter()-t0:.0f}s)")
    except Exception as exc:
        print(f"{facet:20s} FAILED: {exc}")

In [ ]:
themes = fs.graph_store.list_themes()
print(f"themes total: {len(themes)}")
for t in themes[:15]:
    print(f"  {t.theme_id}")

## 11. Layer C — cards

One LLM call per theme. Run facet by facet so a failure costs one facet, not
the whole pass, and so the spend is visible as it happens.

In [ ]:
for facet in FACETS:
    t0 = time.perf_counter()
    try:
        meta = fs.build_layer_c(facet=facet)
        print(f"{facet:20s} {meta.record_count:5d} cards  ({time.perf_counter()-t0:.0f}s)")
    except Exception as exc:
        print(f"{facet:20s} FAILED: {exc}")

es.indices.refresh(index=CARD_INDEX)
print("cards in the index:", count(CARD_INDEX))

## 12. Verify, then publish

The graph is now in Neo4j and Elasticsearch. The FoodScholar API still serves
the *previous* projection until the browse index is rebuilt — which is the
behaviour you want, since a stale graph beats no graph while this runs.

Publish with, as an admin:

```
POST /api/v1/foodscholar/graph/reindex
```

It reads the whole graph and repoints the alias only on success, so a failed
projection leaves the last good one in place.

In [ ]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    NEO4J_URL,
    auth=(os.environ.get("NEO4J_USER", "neo4j"), os.environ["NEO4J_PASSWORD"]),
)
with driver.session() as session:
    for label in ("Shelf", "Theme", "Card", "Entity"):
        n = session.run(f"MATCH (n:{label}) RETURN count(n) AS c").single()["c"]
        print(f"  :{label:8s} {n}")
driver.close()

print()
print(f"  chunks        {count(CHUNK_INDEX)}")
print(f"  cards         {count(CARD_INDEX)}")
print(f"  graph_version {fs.config_hash}")
print("\nNow run POST /api/v1/foodscholar/graph/reindex as an admin to publish.")